# 06: Cross-Ablation — Activation × Attention

**Purpose:** Resolve the FP32 accuracy inversion (TinyYOLO-q 41.2% > TinyYOLO-std 38.7%).

The two variants differ in BOTH activation (SiLU vs ReLU6) AND attention (SE+LightSpatial vs ECA).
This experiment isolates each factor by running all 4 combinations.

| Config | Variant (act) | Attention | Expected baseline |
|--------|--------------|-----------|------------------|
| A | standard (SiLU) | default (SE+Spatial) | = TinyYOLO-std (38.7%) |
| B | standard (SiLU) | eca | NEW |
| C | quantized (ReLU6) | se | NEW |
| D | quantized (ReLU6) | default (ECA) | = TinyYOLO-q (41.2%) |

**Seeds:** 42, 123, 256 (3 runs per config = 12 total)

**Dataset:** Pascal VOC 2007+2012, 300 epochs, 416×416

**Estimated time:** ~63 hours on T4 (~5.25h per run × 12 runs)

**NOTE:** If running on Kaggle with 12h session limits, split across multiple sessions.
Each run saves independently to `experiments/results/`.

In [ ]:
import os, sys, socket
try:
    socket.create_connection(("github.com", 80), timeout=5)
except:
    print("❌ ERROR: No Internet connection. If on Kaggle, toggle 'Internet' to ON in the sidebar settings.")
    sys.exit(1)

!rm -rf tinyYOLO && git clone https://github.com/ShMazumder/tinyYOLO.git
%cd tinyYOLO
!pip install -e . -q
!pip install tqdm timm -q
!python3 -c "from ultralytics.data.utils import check_det_dataset; check_det_dataset('VOC.yaml')"

## Verify attention decoupling works

In [ ]:
# Quick sanity check: all 4 configs build successfully
from tinyYOLO.models import build_model
import torch

configs = [
    ('standard', 'default', 'A: SiLU + SE/Spatial'),
    ('standard', 'eca',     'B: SiLU + ECA'),
    ('quantized', 'se',     'C: ReLU6 + SE/Spatial'),
    ('quantized', 'default','D: ReLU6 + ECA'),
]

x = torch.randn(1, 3, 416, 416)
for variant, attn, label in configs:
    model, info = build_model(task='det', variant=variant, nc=20, attention=attn)
    out = model(x)
    print(f"  ✓ {label}: {info['total_params_M']}M params, "
          f"attn3={model.backbone.attn3.__class__.__name__}, "
          f"attn4={model.backbone.attn4.__class__.__name__}, "
          f"output shapes: {[o.shape for o in out]}")

print("\n  All 4 configs verified! ✓")

## Config A: SiLU + SE/Spatial (= TinyYOLO-std baseline)
This should reproduce ~38.7% mAP@50

In [ ]:
SEEDS = [42, 123, 256]
for s in SEEDS:
    !python3 scripts/train.py --task det --variant standard --attention default \
        --data voc.yaml --imgsz 416 --epochs 300 --batch 128 --seed {s} \
        --warmup 3 --pretrained --compile --name xabl-A-silu-se-seed{s}

## Config B: SiLU + ECA (cross — isolates ECA effect with SiLU)

In [ ]:
SEEDS = [42, 123, 256]
for s in SEEDS:
    !python3 scripts/train.py --task det --variant standard --attention eca \
        --data voc.yaml --imgsz 416 --epochs 300 --batch 128 --seed {s} \
        --warmup 3 --pretrained --compile --name xabl-B-silu-eca-seed{s}

## Config C: ReLU6 + SE/Spatial (cross — isolates ReLU6 effect with SE)

In [ ]:
SEEDS = [42, 123, 256]
for s in SEEDS:
    !python3 scripts/train.py --task det --variant quantized --attention se \
        --data voc.yaml --imgsz 416 --epochs 300 --batch 128 --seed {s} \
        --warmup 3 --pretrained --compile --name xabl-C-relu6-se-seed{s}

## Config D: ReLU6 + ECA (= TinyYOLO-q baseline)
This should reproduce ~41.2% mAP@50

In [ ]:
SEEDS = [42, 123, 256]
for s in SEEDS:
    !python3 scripts/train.py --task det --variant quantized --attention default \
        --data voc.yaml --imgsz 416 --epochs 300 --batch 128 --seed {s} \
        --warmup 3 --pretrained --compile --name xabl-D-relu6-eca-seed{s}

## Results Analysis
Collect results from all 12 runs and compute the 2×2 factorial table.

In [ ]:
import json, glob
from pathlib import Path
import numpy as np

results = {}
for config_id, pattern in [
    ('A (SiLU+SE)',  'xabl-A-silu-se-*'),
    ('B (SiLU+ECA)', 'xabl-B-silu-eca-*'),
    ('C (ReLU6+SE)', 'xabl-C-relu6-se-*'),
    ('D (ReLU6+ECA)','xabl-D-relu6-eca-*'),
]:
    maps = []
    for d in sorted(glob.glob(f'experiments/results/{pattern}')):
        summary = Path(d) / 'summary.json'
        if summary.exists():
            with open(summary) as f:
                data = json.load(f)
            maps.append(data.get('best_map50', 0) * 100)
    results[config_id] = maps

print('=' * 70)
print('CROSS-ABLATION RESULTS: Activation × Attention')
print('=' * 70)
print(f'{"Config":<20} {"Runs":>5} {"mAP@50 (%)":>12} {"Std":>8}')
print('-' * 70)
for cfg, maps in results.items():
    if maps:
        print(f'{cfg:<20} {len(maps):>5} {np.mean(maps):>12.1f} {np.std(maps):>8.2f}')
    else:
        print(f'{cfg:<20} {"N/A":>5} {"—":>12} {"—":>8}')
print('=' * 70)

# Factorial analysis
if all(len(v) >= 2 for v in results.values()):
    a, b, c, d = [np.mean(results[k]) for k in results]
    eca_effect = ((b - a) + (d - c)) / 2
    relu6_effect = ((c - a) + (d - b)) / 2
    interaction = (d - c) - (b - a)
    print(f'\nFactorial Analysis:')
    print(f'  ECA effect (vs SE):     {eca_effect:+.1f}% mAP@50')
    print(f'  ReLU6 effect (vs SiLU): {relu6_effect:+.1f}% mAP@50')
    print(f'  Interaction effect:     {interaction:+.1f}% mAP@50')
    print(f'\n  Total gap explained:    {eca_effect + relu6_effect + interaction:+.1f}%')
    print(f'  Observed gap (D - A):   {d - a:+.1f}%')

    from scipy import stats
    # T-test: is D significantly > A?
    t_stat, p_val = stats.ttest_ind(results['D (ReLU6+ECA)'],
                                     results['A (SiLU+SE)'])
    print(f'\n  T-test (D vs A): t={t_stat:.2f}, p={p_val:.4f}')
    if p_val < 0.05:
        print(f'  → Statistically significant at p<0.05')
    else:
        print(f'  → NOT statistically significant at p<0.05')

## Save consolidated results

In [ ]:
# Save complete results for manuscript
output = {
    'experiment': 'cross_ablation_activation_attention',
    'dataset': 'VOC 2007+2012',
    'epochs': 300,
    'imgsz': 416,
    'seeds': [42, 123, 256],
    'results': {k: {'maps': v, 'mean': float(np.mean(v)), 'std': float(np.std(v))}
                for k, v in results.items() if v},
}

out_path = Path('experiments/results/cross_ablation_summary.json')
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f'Saved to {out_path}')